# RealNVP on make_circles – Concentric Circles

**Task:**
- Create a dataset with `sklearn.datasets.make_circles` (two concentric circles)
- Build a RealNVP network to learn the data distribution
- Train the model on this dataset
- Generate points with the trained model and compare them to the original data
- Evaluate quality using KL-divergence (`tf.keras.losses.KLDivergence`)

## 1. Imports and configuration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp
from sklearn import datasets
from tensorflow.keras import (
    layers,
    models,
    regularizers,
    metrics,
    optimizers,
    callbacks,
)

print("TensorFlow:", tf.__version__)

## 2. Dataset creation (make_circles)

In [ ]:
# Two concentric circles with make_circles
n_samples = 30000
noise = 0.05
data, labels = datasets.make_circles(n_samples, noise=noise, factor=0.5, random_state=42)
data = data.astype("float32")

# Normalization (as in the make_moons notebook)
norm = layers.Normalization()
norm.adapt(data)
normalized_data = norm(data)

plt.figure(figsize=(6, 5))
plt.scatter(normalized_data.numpy()[:, 0], normalized_data.numpy()[:, 1], c=labels, cmap="viridis", s=1, alpha=0.7)
plt.title("make_circles data (normalized)")
plt.xlabel("x_1")
plt.ylabel("x_2")
plt.colorbar(label="class")
plt.show()

## 3. RealNVP architecture – Coupling layer and model

In [ ]:
INPUT_DIM = 2
COUPLING_DIM = 256
COUPLING_LAYERS = 6
REGULARIZATION = 0.01
BATCH_SIZE = 256
EPOCHS = 100

def Coupling(input_dim, coupling_dim, reg):
    input_layer = layers.Input(shape=(input_dim,))
    s_layer_1 = layers.Dense(coupling_dim, activation="relu", kernel_regularizer=regularizers.l2(reg))(input_layer)
    s_layer_2 = layers.Dense(coupling_dim, activation="relu", kernel_regularizer=regularizers.l2(reg))(s_layer_1)
    s_layer_3 = layers.Dense(coupling_dim, activation="relu", kernel_regularizer=regularizers.l2(reg))(s_layer_2)
    s_layer_4 = layers.Dense(coupling_dim, activation="relu", kernel_regularizer=regularizers.l2(reg))(s_layer_3)
    s_layer_5 = layers.Dense(input_dim, activation="tanh", kernel_regularizer=regularizers.l2(reg))(s_layer_4)
    t_layer_1 = layers.Dense(coupling_dim, activation="relu", kernel_regularizer=regularizers.l2(reg))(input_layer)
    t_layer_2 = layers.Dense(coupling_dim, activation="relu", kernel_regularizer=regularizers.l2(reg))(t_layer_1)
    t_layer_3 = layers.Dense(coupling_dim, activation="relu", kernel_regularizer=regularizers.l2(reg))(t_layer_2)
    t_layer_4 = layers.Dense(coupling_dim, activation="relu", kernel_regularizer=regularizers.l2(reg))(t_layer_3)
    t_layer_5 = layers.Dense(input_dim, activation="linear", kernel_regularizer=regularizers.l2(reg))(t_layer_4)
    return models.Model(inputs=input_layer, outputs=[s_layer_5, t_layer_5])

In [ ]:
class RealNVP(models.Model):
    def __init__(self, input_dim, coupling_layers, coupling_dim, regularization):
        super(RealNVP, self).__init__()
        self.coupling_layers = coupling_layers
        self.distribution = tfp.distributions.MultivariateNormalDiag(
            loc=[0.0, 0.0], scale_diag=[1.0, 1.0]
        )
        self.masks = np.array(
            [[0, 1], [1, 0]] * (coupling_layers // 2), dtype="float32"
        )
        self.loss_tracker = metrics.Mean(name="loss")
        self.layers_list = [
            Coupling(input_dim, coupling_dim, regularization)
            for _ in range(coupling_layers)
        ]

    @property
    def metrics(self):
        return [self.loss_tracker]

    def call(self, x, training=True):
        log_det_inv = 0
        direction = 1
        if training:
            direction = -1
        for i in range(self.coupling_layers)[::direction]:
            x_masked = x * self.masks[i]
            reversed_mask = 1 - self.masks[i]
            s, t = self.layers_list[i](x_masked)
            s *= reversed_mask
            t *= reversed_mask
            gate = (direction - 1) / 2
            x = (
                reversed_mask
                * (x * tf.exp(direction * s) + direction * t * tf.exp(gate * s))
                + x_masked
            )
            log_det_inv += gate * tf.reduce_sum(s, axis=1)
        return x, log_det_inv

    def log_loss(self, x):
        y, logdet = self(x)
        log_likelihood = self.distribution.log_prob(y) + logdet
        return -tf.reduce_mean(log_likelihood)

    def train_step(self, data):
        with tf.GradientTape() as tape:
            loss = self.log_loss(data)
        g = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(g, self.trainable_variables))
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def test_step(self, data):
        loss = self.log_loss(data)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}


model = RealNVP(
    input_dim=INPUT_DIM,
    coupling_layers=COUPLING_LAYERS,
    coupling_dim=COUPLING_DIM,
    regularization=REGULARIZATION,
)

## 4. Training the RealNVP model

In [ ]:
model.compile(optimizer=optimizers.Adam(learning_rate=0.0001))

history = model.fit(
    normalized_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], color="blue")
plt.title("Loss (NLL) during training")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()

## 5. Point generation and comparison with original data

In [ ]:
num_samples = 3000
z_samples = model.distribution.sample(num_samples)
generated, _ = model.predict(z_samples, verbose=0)

# Data -> latent space (for visualization)
z_data, _ = model(normalized_data)
z_data = z_data.numpy()
generated = np.array(generated)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))

axes[0, 0].scatter(normalized_data[:, 0], normalized_data[:, 1], c="green", s=1, alpha=0.5)
axes[0, 0].set_title("Original data (X space)")
axes[0, 0].set_xlabel("x_1")
axes[0, 0].set_ylabel("x_2")
axes[0, 0].set_xlim([-2, 2])
axes[0, 0].set_ylim([-2, 2])

axes[0, 1].scatter(z_data[:, 0], z_data[:, 1], c="green", s=1, alpha=0.5)
axes[0, 1].set_title("f(X) – data in latent space Z")
axes[0, 1].set_xlabel("z_1")
axes[0, 1].set_ylabel("z_2")
axes[0, 1].set_xlim([-3, 3])
axes[0, 1].set_ylim([-3, 3])

axes[1, 0].scatter(z_samples[:, 0], z_samples[:, 1], c="blue", s=1, alpha=0.5)
axes[1, 0].set_title("Samples Z ~ N(0,I)")
axes[1, 0].set_xlabel("z_1")
axes[1, 0].set_ylabel("z_2")
axes[1, 0].set_xlim([-3, 3])
axes[1, 0].set_ylim([-3, 3])

axes[1, 1].scatter(generated[:, 0], generated[:, 1], c="blue", s=1, alpha=0.5)
axes[1, 1].set_title("g(Z) – generated points (X space)")
axes[1, 1].set_xlabel("x_1")
axes[1, 1].set_ylabel("x_2")
axes[1, 1].set_xlim([-2, 2])
axes[1, 1].set_ylim([-2, 2])

plt.suptitle("RealNVP: data vs generation (make_circles)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Direct comparison: original vs generated on the same plot
plt.figure(figsize=(7, 6))
plt.scatter(normalized_data[:, 0], normalized_data[:, 1], c="green", s=1, alpha=0.4, label="Original data")
plt.scatter(generated[:, 0], generated[:, 1], c="blue", s=1, alpha=0.4, label="Generated by RealNVP")
plt.title("Comparison: original data vs generated points")
plt.xlabel("x_1")
plt.ylabel("x_2")
plt.legend()
plt.xlim([-2, 2])
plt.ylim([-2, 2])
plt.show()

## 6. Evaluation by KL-divergence

We discretize the plane into a grid (2D histogram) to obtain two probability distributions (real vs generated data), then compute KL-divergence using `tf.keras.losses.KLDivergence`.

In [ ]:
def histogram2d_to_proba(points, bins=30, x_range=(-2, 2), y_range=(-2, 2)):
    """Normalized 2D histogram -> probability distribution (flat vector)."""
    h, _, _ = np.histogram2d(
        points[:, 0], points[:, 1],
        bins=bins, range=[x_range, y_range]
    )
    total = h.sum()
    if total <= 0:
        total = 1.0
    p = (h / total).flatten()
    return p

def kl_divergence_binned(real_points, gen_points, bins=30, eps=1e-10):
    """
    KL(P || Q) where P = real distribution, Q = generated distribution.
    Uses tf.keras.losses.KLDivergence.
    """
    p_real = histogram2d_to_proba(real_points, bins=bins)
    p_gen = histogram2d_to_proba(gen_points, bins=bins)
    p_real = np.clip(p_real, eps, 1.0)
    p_gen = np.clip(p_gen, eps, 1.0)
    # Re-normalize after clip
    p_real = p_real / p_real.sum()
    p_gen = p_gen / p_gen.sum()
    p_real = p_real.astype(np.float32).reshape(1, -1)
    p_gen = p_gen.astype(np.float32).reshape(1, -1)
    kl = tf.keras.losses.KLDivergence()(p_real, p_gen)
    return kl.numpy()

n_bins = 40
kl_value = kl_divergence_binned(
    normalized_data.numpy(),
    generated,
    bins=n_bins,
)
print(f"KL-divergence (P_real || P_generated), {n_bins}x{n_bins} bins: {kl_value:.6f}")

In [ ]:
# Display 2D histograms to illustrate P and Q
p_real = histogram2d_to_proba(normalized_data.numpy(), bins=n_bins).reshape(n_bins, n_bins)
p_gen = histogram2d_to_proba(generated, bins=n_bins).reshape(n_bins, n_bins)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(p_real, origin="lower", extent=[-2, 2, -2, 2], aspect="auto", cmap="viridis")
axes[0].set_title("Real distribution (2D histogram)")
axes[0].set_xlabel("x_1")
axes[0].set_ylabel("x_2")
axes[1].imshow(p_gen, origin="lower", extent=[-2, 2, -2, 2], aspect="auto", cmap="viridis")
axes[1].set_title("Generated distribution (2D histogram)")
axes[1].set_xlabel("x_1")
axes[1].set_ylabel("x_2")
plt.suptitle(f"KL(P_real || P_generated) = {kl_value:.4f}")
plt.tight_layout()
plt.show()